<a href="https://colab.research.google.com/github/duruamobi/AAI2026/blob/main/Agentic_AI_in_Supply_Chain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
from dataclasses import dataclass
from pathlib import Path
from statistics import NormalDist
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd


# ============================================================
# Inventory Replenishment Agent
# ------------------------------------------------------------
# This script:
# 1) Auto-generates sample CSV files if they do not exist
# 2) Loads and prepares daily sales, inventory, and SKU params
# 3) Builds an EWMA demand forecaster per SKU
# 4) Computes safety stock from forecast error + service level
# 5) Runs an agent loop that places purchase orders or waits
# 6) Simulates outcomes and compares against a baseline policy
# 7) Prints daily decision logs with rationale
#
# Rubric coverage:
# - Agent goal: minimize stockouts and total cost
# - Inputs: forecast, inventory, lead time, service level, costs
# - Outputs: place order quantity or wait
# - Guardrails: min order qty, lead time, safety stock, inventory cap
# - Metrics: stockouts, fill rate, holding cost, stockout cost, total cost
# - Transparency: daily logs explain each decision
# ============================================================


# -----------------------------
# Configuration
# -----------------------------
DATA_DIR = Path(".")
SALES_PATH = DATA_DIR / "sales.csv"
INVENTORY_PATH = DATA_DIR / "inventory.csv"
PARAMS_PATH = DATA_DIR / "params.csv"

DEFAULT_ALPHA = 0.35
REVIEW_PERIOD_DAYS = 1
ORDER_COST = 0.0
RANDOM_SEED = 42
PRINT_AGENT_LOG = True


# -----------------------------
# Data classes
# -----------------------------
@dataclass
class SKUParams:
    sku: str
    unit_cost: float
    holding_cost_per_day: float
    stockout_cost: float
    lead_time_days: int
    min_order_qty: int
    service_level: float
    inventory_cap_days: int = 45  # simple guardrail against unrealistic over-ordering


@dataclass
class PendingOrder:
    sku: str
    qty: int
    arrival_date: pd.Timestamp


# -----------------------------
# Sample data generation
# -----------------------------
def generate_sample_data(
    sales_path: Path,
    inventory_path: Path,
    params_path: Path,
    n_days: int = 90,
    skus: List[str] = None,
) -> None:
    """Auto-generate 1-3 SKU example data if files do not exist."""
    if skus is None:
        skus = ["SKU_A", "SKU_B", "SKU_C"]

    rng = np.random.default_rng(RANDOM_SEED)
    start_date = pd.Timestamp("2025-01-01")
    dates = pd.date_range(start_date, periods=n_days, freq="D")

    # Different demand profiles per SKU
    demand_profiles = {
        "SKU_A": {"base": 18, "trend": 0.03, "dow_boost": [0, 0, 1, 1, 2, 4, 3]},
        "SKU_B": {"base": 9, "trend": 0.01, "dow_boost": [0, 1, 0, 1, 1, 2, 2]},
        "SKU_C": {"base": 25, "trend": -0.01, "dow_boost": [1, 1, 2, 2, 3, 5, 4]},
    }

    sales_rows = []
    for sku in skus:
        profile = demand_profiles.get(sku, {"base": 12, "trend": 0.0, "dow_boost": [0] * 7})
        for i, date in enumerate(dates):
            mean_demand = max(
                1,
                profile["base"] + i * profile["trend"] + profile["dow_boost"][date.dayofweek],
            )
            qty_sold = int(rng.poisson(lam=mean_demand))
            sales_rows.append({"date": date.strftime("%Y-%m-%d"), "sku": sku, "qty_sold": qty_sold})

    sales_df = pd.DataFrame(sales_rows)

    inventory_df = pd.DataFrame(
        {
            "sku": skus,
            "opening_stock": [140, 80, 170][: len(skus)],
        }
    )

    params_df = pd.DataFrame(
        {
            "sku": skus,
            "unit_cost": [12.0, 20.0, 8.0][: len(skus)],
            "holding_cost_per_day": [0.04, 0.06, 0.03][: len(skus)],
            "stockout_cost": [9.0, 15.0, 6.0][: len(skus)],
            "lead_time_days": [5, 7, 4][: len(skus)],
            "min_order_qty": [40, 30, 50][: len(skus)],
            "service_level": [0.95, 0.97, 0.92][: len(skus)],
        }
    )

    sales_df.to_csv(sales_path, index=False)
    inventory_df.to_csv(inventory_path, index=False)
    params_df.to_csv(params_path, index=False)


# -----------------------------
# Loading and preparation
# -----------------------------
def ensure_data_files() -> None:
    if not (SALES_PATH.exists() and INVENTORY_PATH.exists() and PARAMS_PATH.exists()):
        print("CSV files not found. Auto-generating sample supply chain data...")
        generate_sample_data(SALES_PATH, INVENTORY_PATH, PARAMS_PATH)


def load_and_prepare_data(
    sales_path: Path,
    inventory_path: Path,
    params_path: Path,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, SKUParams]]:
    """Load CSVs and aggregate daily demand per SKU."""
    sales = pd.read_csv(sales_path, parse_dates=["date"])
    inventory = pd.read_csv(inventory_path)
    params = pd.read_csv(params_path)

    required_sales_cols = {"date", "sku", "qty_sold"}
    required_inventory_cols = {"sku", "opening_stock"}
    required_params_cols = {
        "sku",
        "unit_cost",
        "holding_cost_per_day",
        "stockout_cost",
        "lead_time_days",
        "min_order_qty",
        "service_level",
    }

    assert required_sales_cols.issubset(sales.columns), "sales.csv missing required columns"
    assert required_inventory_cols.issubset(inventory.columns), "inventory.csv missing required columns"
    assert required_params_cols.issubset(params.columns), "params.csv missing required columns"

    # Aggregate demand per day per SKU and fill missing dates with zero demand.
    daily_sales = (
        sales.groupby(["date", "sku"], as_index=False)["qty_sold"]
        .sum()
        .sort_values(["sku", "date"])
    )

    all_dates = pd.date_range(daily_sales["date"].min(), daily_sales["date"].max(), freq="D")
    all_skus = sorted(daily_sales["sku"].unique())
    full_index = pd.MultiIndex.from_product([all_dates, all_skus], names=["date", "sku"])

    daily_sales = (
        daily_sales.set_index(["date", "sku"])
        .reindex(full_index, fill_value=0)
        .reset_index()
        .sort_values(["sku", "date"])
    )

    params_map = {
        row["sku"]: SKUParams(
            sku=row["sku"],
            unit_cost=float(row["unit_cost"]),
            holding_cost_per_day=float(row["holding_cost_per_day"]),
            stockout_cost=float(row["stockout_cost"]),
            lead_time_days=int(row["lead_time_days"]),
            min_order_qty=int(row["min_order_qty"]),
            service_level=float(row["service_level"]),
        )
        for _, row in params.iterrows()
    }

    return daily_sales, inventory, params_map


# -----------------------------
# Forecasting and safety stock
# -----------------------------
def ewma_forecast(history: List[float], alpha: float = DEFAULT_ALPHA) -> float:
    """Simple EWMA forecast. Uses the first observation as initial forecast."""
    if not history:
        return 0.0
    forecast = float(history[0])
    for actual in history[1:]:
        forecast = alpha * float(actual) + (1 - alpha) * forecast
    return max(0.0, forecast)


def naive_forecast(history: List[float]) -> float:
    """Naive forecast: tomorrow equals yesterday."""
    if not history:
        return 0.0
    return max(0.0, float(history[-1]))


def forecast_error_std(history: List[float], alpha: float = DEFAULT_ALPHA) -> float:
    """Estimate one-step forecast error std from historical EWMA errors."""
    if len(history) < 3:
        if len(history) == 0:
            return 1.0
        return max(1.0, float(np.std(history, ddof=0)))

    preds = []
    actuals = []
    running_history = [history[0]]
    for actual in history[1:]:
        preds.append(ewma_forecast(running_history, alpha))
        actuals.append(actual)
        running_history.append(actual)

    errors = np.array(actuals) - np.array(preds)
    return max(1.0, float(np.std(errors, ddof=1)))


def compute_safety_stock(
    daily_error_std: float,
    service_level: float,
    lead_time_days: int,
    review_period_days: int,
) -> float:
    """Safety stock based on z-score and demand uncertainty over protection period."""
    z = NormalDist().inv_cdf(service_level)
    protection_period = max(1, lead_time_days + review_period_days)
    return max(0.0, z * daily_error_std * math.sqrt(protection_period))


# -----------------------------
# Agent logic
# -----------------------------
def pipeline_inventory(
    pending_orders: List[PendingOrder],
    sku: str,
    as_of_date: pd.Timestamp,
    horizon_days: int,
) -> int:
    horizon_end = as_of_date + pd.Timedelta(days=horizon_days)
    return sum(
        po.qty
        for po in pending_orders
        if po.sku == sku and as_of_date < po.arrival_date <= horizon_end
    )


def due_today_qty(pending_orders: List[PendingOrder], sku: str, today: pd.Timestamp) -> int:
    return sum(po.qty for po in pending_orders if po.sku == sku and po.arrival_date == today)


def remove_received_orders(pending_orders: List[PendingOrder], today: pd.Timestamp) -> List[PendingOrder]:
    return [po for po in pending_orders if po.arrival_date != today]


def recommended_order_qty(
    sku: str,
    current_stock: int,
    demand_history: List[float],
    params: SKUParams,
    pending_orders: List[PendingOrder],
    today: pd.Timestamp,
    alpha: float,
    review_period_days: int,
    use_naive: bool = False,
) -> Tuple[int, Dict[str, float]]:
    """
    Order-up-to style logic:
    target_position = expected demand over protection period + safety stock
    inventory_position = on-hand + inbound during protection period
    place order if inventory position is below target.
    """
    forecast = naive_forecast(demand_history) if use_naive else ewma_forecast(demand_history, alpha)
    error_std = forecast_error_std(demand_history, alpha)
    safety_stock = compute_safety_stock(
        daily_error_std=error_std,
        service_level=params.service_level,
        lead_time_days=params.lead_time_days,
        review_period_days=review_period_days,
    )

    protection_days = params.lead_time_days + review_period_days
    expected_demand = forecast * protection_days
    inbound = pipeline_inventory(pending_orders, sku, today, protection_days)
    inventory_position = current_stock + inbound
    raw_target = expected_demand + safety_stock

    # Guardrail: cap target to avoid unrealistic over-ordering.
    inventory_cap = forecast * params.inventory_cap_days + safety_stock
    target_position = min(raw_target, inventory_cap)

    reorder_gap = target_position - inventory_position
    if reorder_gap <= 0:
        order_qty = 0
    else:
        order_qty = max(params.min_order_qty, int(math.ceil(reorder_gap)))

    details = {
        "forecast": round(forecast, 2),
        "error_std": round(error_std, 2),
        "safety_stock": round(safety_stock, 2),
        "expected_demand": round(expected_demand, 2),
        "inbound_within_horizon": inbound,
        "inventory_position": round(inventory_position, 2),
        "target_position": round(target_position, 2),
        "reorder_gap": round(reorder_gap, 2),
    }
    return order_qty, details


# -----------------------------
# Simulation
# -----------------------------
def simulate_policy(
    daily_sales: pd.DataFrame,
    inventory_df: pd.DataFrame,
    params_map: Dict[str, SKUParams],
    alpha: float = DEFAULT_ALPHA,
    review_period_days: int = REVIEW_PERIOD_DAYS,
    order_cost: float = ORDER_COST,
    use_naive: bool = False,
    policy_name: str = "agent",
    baseline_mode: str = "none",
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    baseline_mode options:
      - 'none' => agent with EWMA/naive logic
      - 'fixed_reorder' => simple baseline using reorder point from avg demand only
    """
    daily_sales = daily_sales.copy().sort_values(["date", "sku"])
    sku_list = sorted(daily_sales["sku"].unique())
    dates = sorted(daily_sales["date"].unique())

    stock = {row["sku"]: int(row["opening_stock"]) for _, row in inventory_df.iterrows()}
    history = {sku: [] for sku in sku_list}
    pending_orders: List[PendingOrder] = []

    logs = []
    metrics_rows = []

    cumulative = {
        sku: {
            "demand": 0,
            "fulfilled": 0,
            "stockout_units": 0,
            "holding_cost": 0.0,
            "stockout_cost": 0.0,
            "order_cost": 0.0,
            "orders_placed": 0,
        }
        for sku in sku_list
    }

    sales_lookup = {
        (row["date"], row["sku"]): int(row["qty_sold"])
        for _, row in daily_sales.iterrows()
    }

    for today in dates:
        # Receive orders due today first
        for sku in sku_list:
            received_qty = due_today_qty(pending_orders, sku, today)
            if received_qty > 0:
                stock[sku] += received_qty
                logs.append(
                    {
                        "date": today,
                        "sku": sku,
                        "action": "RECEIVE_PO",
                        "qty": received_qty,
                        "reason": f"Received inbound order due today after lead time.",
                    }
                )
        pending_orders = remove_received_orders(pending_orders, today)

        # Agent decides whether to order
        for sku in sku_list:
            params = params_map[sku]
            current_history = history[sku][:] if history[sku] else [0]

            if baseline_mode == "fixed_reorder":
                avg_demand = float(np.mean(current_history)) if current_history else 0.0
                reorder_point = avg_demand * (params.lead_time_days + review_period_days)
                reorder_point += 0.5 * avg_demand * math.sqrt(max(1, params.lead_time_days))
                inbound = pipeline_inventory(pending_orders, sku, today, params.lead_time_days + review_period_days)
                inventory_position = stock[sku] + inbound
                gap = reorder_point - inventory_position
                order_qty = 0 if gap <= 0 else max(params.min_order_qty, int(math.ceil(gap)))
                details = {
                    "forecast": round(avg_demand, 2),
                    "error_std": None,
                    "safety_stock": round(0.5 * avg_demand * math.sqrt(max(1, params.lead_time_days)), 2),
                    "expected_demand": round(avg_demand * (params.lead_time_days + review_period_days), 2),
                    "inbound_within_horizon": inbound,
                    "inventory_position": round(inventory_position, 2),
                    "target_position": round(reorder_point, 2),
                    "reorder_gap": round(gap, 2),
                }
            else:
                order_qty, details = recommended_order_qty(
                    sku=sku,
                    current_stock=stock[sku],
                    demand_history=current_history,
                    params=params,
                    pending_orders=pending_orders,
                    today=today,
                    alpha=alpha,
                    review_period_days=review_period_days,
                    use_naive=use_naive,
                )

            if order_qty > 0:
                arrival_date = today + pd.Timedelta(days=params.lead_time_days)
                pending_orders.append(PendingOrder(sku=sku, qty=order_qty, arrival_date=arrival_date))
                cumulative[sku]["order_cost"] += order_cost
                cumulative[sku]["orders_placed"] += 1
                reason = (
                    f"Projected inventory position {details['inventory_position']} is below target {details['target_position']}. "
                    f"Forecast={details['forecast']}/day, safety_stock={details['safety_stock']}, "
                    f"lead_time={params.lead_time_days}, MOQ={params.min_order_qty}."
                )
                logs.append(
                    {
                        "date": today,
                        "sku": sku,
                        "action": "PLACE_PO",
                        "qty": order_qty,
                        "reason": reason,
                    }
                )
            else:
                reason = (
                    f"Wait: inventory position {details['inventory_position']} covers target {details['target_position']}. "
                    f"Ordering now would add holding cost without reducing near-term stockout risk enough."
                )
                logs.append(
                    {
                        "date": today,
                        "sku": sku,
                        "action": "WAIT",
                        "qty": 0,
                        "reason": reason,
                    }
                )

        # Realized demand and end-of-day cost tracking
        for sku in sku_list:
            params = params_map[sku]
            demand = sales_lookup[(today, sku)]
            fulfilled = min(stock[sku], demand)
            stockout_units = max(0, demand - stock[sku])
            ending_stock = max(0, stock[sku] - demand)

            stock[sku] = ending_stock
            history[sku].append(demand)

            cumulative[sku]["demand"] += demand
            cumulative[sku]["fulfilled"] += fulfilled
            cumulative[sku]["stockout_units"] += stockout_units
            cumulative[sku]["holding_cost"] += ending_stock * params.holding_cost_per_day
            cumulative[sku]["stockout_cost"] += stockout_units * params.stockout_cost

            metrics_rows.append(
                {
                    "date": today,
                    "sku": sku,
                    "policy": policy_name,
                    "demand": demand,
                    "fulfilled": fulfilled,
                    "stockout_units": stockout_units,
                    "ending_stock": ending_stock,
                    "holding_cost": ending_stock * params.holding_cost_per_day,
                    "stockout_cost": stockout_units * params.stockout_cost,
                }
            )

    summary_rows = []
    for sku in sku_list:
        d = cumulative[sku]["demand"]
        f = cumulative[sku]["fulfilled"]
        fill_rate = (f / d) if d > 0 else 1.0
        total_cost = (
            cumulative[sku]["holding_cost"]
            + cumulative[sku]["stockout_cost"]
            + cumulative[sku]["order_cost"]
        )
        summary_rows.append(
            {
                "sku": sku,
                "policy": policy_name,
                "total_demand": d,
                "fulfilled_units": f,
                "stockout_units": cumulative[sku]["stockout_units"],
                "fill_rate": round(fill_rate, 4),
                "orders_placed": cumulative[sku]["orders_placed"],
                "holding_cost": round(cumulative[sku]["holding_cost"], 2),
                "stockout_cost": round(cumulative[sku]["stockout_cost"], 2),
                "order_cost": round(cumulative[sku]["order_cost"], 2),
                "total_cost": round(total_cost, 2),
            }
        )

    log_df = pd.DataFrame(logs)
    summary_df = pd.DataFrame(summary_rows)
    daily_metrics_df = pd.DataFrame(metrics_rows)
    return log_df, pd.merge(summary_df, daily_metrics_df.groupby(["sku", "policy"], as_index=False).size(), on=["sku", "policy"], how="left")


# -----------------------------
# Reporting helpers
# -----------------------------
def print_agent_log(log_df: pd.DataFrame, max_rows: int = 120) -> None:
    print("\n" + "=" * 90)
    print("DAILY AGENT LOG")
    print("=" * 90)
    display_df = log_df.copy()
    if len(display_df) > max_rows:
        display_df = display_df.head(max_rows)
        print(f"Showing first {max_rows} rows of {len(log_df)} total log rows...\n")
    for _, row in display_df.iterrows():
        print(
            f"{row['date'].date()} | {row['sku']:>5} | {row['action']:<10} | "
            f"qty={int(row['qty']):>4} | {row['reason']}"
        )


def compare_policies(agent_summary: pd.DataFrame, baseline_summary: pd.DataFrame) -> pd.DataFrame:
    merged = agent_summary.merge(
        baseline_summary,
        on="sku",
        suffixes=("_agent", "_baseline"),
    )
    comparison_rows = []
    for _, row in merged.iterrows():
        comparison_rows.append(
            {
                "sku": row["sku"],
                "agent_fill_rate": row["fill_rate_agent"],
                "baseline_fill_rate": row["fill_rate_baseline"],
                "agent_stockout_units": row["stockout_units_agent"],
                "baseline_stockout_units": row["stockout_units_baseline"],
                "agent_total_cost": row["total_cost_agent"],
                "baseline_total_cost": row["total_cost_baseline"],
                "fill_rate_delta": round(row["fill_rate_agent"] - row["fill_rate_baseline"], 4),
                "cost_delta": round(row["total_cost_agent"] - row["total_cost_baseline"], 2),
            }
        )
    return pd.DataFrame(comparison_rows)


def design_note() -> str:
    return """
DESIGN NOTE (aligned to rubric)
--------------------------------
Agent role:
This inventory replenishment agent minimizes stockouts and total cost by using demand forecasts,
current stock, lead time, service level, and costs to decide whether to place a purchase order or wait.

Inputs:
- Daily historical demand by SKU
- Opening stock by SKU
- Unit cost, holding cost, stockout cost
- Lead time, minimum order quantity, and service level

Outputs:
- Order quantity for each SKU each day, or WAIT
- Transparent daily rationale log

Guardrails and business rules:
- Respects minimum order quantity
- Includes lead time when projecting inventory risk
- Uses service level to compute safety stock from forecast error
- Applies an inventory cap to avoid unrealistic over-ordering
- Balances stockout risk against holding cost

Metrics:
- Stockout units
- Fill rate = fulfilled demand / total demand
- Total cost = holding cost + stockout cost + order cost

Ethics, trust, and business responsibility:
- High fill rate supports customer trust and service reliability
- Avoiding excess inventory supports cost responsibility and working capital discipline
- The daily log makes recommendations explainable and auditable
""".strip()


def scaling_note() -> str:
    return """
SCALING NOTE
------------
Deployment:
Run this as a scheduled containerized job (for example, daily in a cloud scheduler + container service).
It can read fresh sales and inventory snapshots from object storage or a data warehouse, write PO recommendations
back to a database, and expose a dashboard for planners.

Monitoring:
Track fill rate, stockout units, forecast error (MAE or RMSE), late PO receipts, and inventory days on hand.
Set alerts when fill rate drops below target, forecast error spikes, or stockouts increase for a SKU.

Reliability:
Add schema checks, missing-data imputation, retries for data ingestion, and defensive defaults when a SKU has
limited history. Keep all decisions logged for auditability and recovery.

Cost controls:
Use inventory caps, working-capital limits, and SKU-level budget thresholds. Add approval rules for unusually
large orders and periodic review of service-level targets versus margin and stockout cost.
""".strip()


# -----------------------------
# Main
# -----------------------------
def main() -> None:
    ensure_data_files()
    daily_sales, inventory_df, params_map = load_and_prepare_data(
        SALES_PATH, INVENTORY_PATH, PARAMS_PATH
    )

    print("Loaded data successfully.")
    print(f"Sales rows: {len(daily_sales)}")
    print(f"Inventory rows: {len(inventory_df)}")
    print(f"Param rows: {len(params_map)}")
    print("\nAggregated daily demand sample:")
    print(daily_sales.head(12).to_string(index=False))

    print("\n" + design_note())

    # Agent policy: EWMA-based replenishment
    agent_log, agent_summary = simulate_policy(
        daily_sales=daily_sales,
        inventory_df=inventory_df,
        params_map=params_map,
        alpha=DEFAULT_ALPHA,
        review_period_days=REVIEW_PERIOD_DAYS,
        order_cost=ORDER_COST,
        use_naive=False,
        policy_name="ewma_agent",
        baseline_mode="none",
    )

    # Baseline policy: simple average-demand fixed reorder logic
    baseline_log, baseline_summary = simulate_policy(
        daily_sales=daily_sales,
        inventory_df=inventory_df,
        params_map=params_map,
        alpha=DEFAULT_ALPHA,
        review_period_days=REVIEW_PERIOD_DAYS,
        order_cost=ORDER_COST,
        use_naive=False,
        policy_name="fixed_reorder_baseline",
        baseline_mode="fixed_reorder",
    )

    if PRINT_AGENT_LOG:
        print_agent_log(agent_log, max_rows=150)

    print("\n" + "=" * 90)
    print("AGENT SUMMARY")
    print("=" * 90)
    print(agent_summary.to_string(index=False))

    print("\n" + "=" * 90)
    print("BASELINE SUMMARY")
    print("=" * 90)
    print(baseline_summary.to_string(index=False))

    comparison = compare_policies(agent_summary, baseline_summary)
    print("\n" + "=" * 90)
    print("POLICY COMPARISON")
    print("=" * 90)
    print(comparison.to_string(index=False))

    print("\nTrade-off reflection:")
    print(
        "Higher service levels increase safety stock, which usually improves fill rate and reduces stockouts, "
        "but also increases holding cost. Lower inventory reduces carrying cost, but can damage service reliability "
        "and customer trust when lead times or demand variability create stockouts."
    )

    print("\n" + scaling_note())


if __name__ == "__main__":
    main()


CSV files not found. Auto-generating sample supply chain data...
Loaded data successfully.
Sales rows: 270
Inventory rows: 3
Param rows: 3

Aggregated daily demand sample:
      date   sku  qty_sold
2025-01-01 SKU_A        23
2025-01-02 SKU_A        24
2025-01-03 SKU_A        14
2025-01-04 SKU_A        20
2025-01-05 SKU_A        23
2025-01-06 SKU_A        17
2025-01-07 SKU_A        19
2025-01-08 SKU_A        24
2025-01-09 SKU_A        23
2025-01-10 SKU_A        24
2025-01-11 SKU_A        22
2025-01-12 SKU_A        16

DESIGN NOTE (aligned to rubric)
--------------------------------
Agent role:
This inventory replenishment agent minimizes stockouts and total cost by using demand forecasts,
current stock, lead time, service level, and costs to decide whether to place a purchase order or wait.

Inputs:
- Daily historical demand by SKU
- Opening stock by SKU
- Unit cost, holding cost, stockout cost
- Lead time, minimum order quantity, and service level

Outputs:
- Order quantity for each S